In [1]:
from pathlib import Path
import json
import time
import pandas as pd

In [2]:
project_root = Path.cwd().parent

benchmark_root = project_root / "data" / "benchmark"
ocr_results_root = project_root / "results" / "ocr"

ocr_results_root.mkdir(
    parents=True,
    exist_ok=True
)

print("Project:", project_root)
print("Benchmark:", benchmark_root)
print("OCR results:", ocr_results_root)

Project: D:\mycode\GlyphForge
Benchmark: D:\mycode\GlyphForge\data\benchmark
OCR results: D:\mycode\GlyphForge\results\ocr


In [3]:
categories = {
    "scanning": "Real5-OmniDocBench-Scanning",
    "warping": "Real5-OmniDocBench-Warping",
    "screen_photography": "Real5-OmniDocBench-Screen-Photography",
    "illumination": "Real5-OmniDocBench-Illumination",
    "skew": "Real5-OmniDocBench-Skew",
}

for category, folder in categories.items():
    path = benchmark_root / folder
    
    files = [
        file
        for file in path.iterdir()
        if file.is_file()
        and file.suffix.lower() in {
            ".png",
            ".jpg",
            ".jpeg",
            ".webp"
        }
    ]
    
    print(category, len(files))

scanning 5
warping 5
screen_photography 5
illumination 5
skew 5


In [4]:
selected_images = {}

for category, folder in categories.items():
    folder_path = benchmark_root / folder
    
    files = sorted([
        file
        for file in folder_path.iterdir()
        if file.is_file()
        and file.suffix.lower() in {
            ".png",
            ".jpg",
            ".jpeg",
            ".webp"
        }
    ])
    
    if not files:
        raise FileNotFoundError(folder_path)
    
    selected_images[category] = files[0]

for category, path in selected_images.items():
    print(category, "->", path.name)

scanning -> PPT_1001115_eng_page_003.png
warping -> PPT_1001115_eng_page_003.png
screen_photography -> PPT_1001115_eng_page_003.png
illumination -> PPT_1001115_eng_page_003.png
skew -> PPT_1001115_eng_page_003.png


In [5]:
from paddleocr import PaddleOCRVL

print("PaddleOCR-VL import successful")

D:\mycode\GlyphForge\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PaddleOCR-VL import successful


In [6]:
pipeline = PaddleOCRVL(
    pipeline_version="v1.6"
)

print("PaddleOCR-VL 1.6 pipeline created")

D:\mycode\GlyphForge\.venv\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-DocLayoutV3', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\soura\.paddlex\official_models\PP-DocLayoutV3`.
Creating model: ('PaddleOCR-VL-1.6-0.9B', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\soura\.paddlex\official_models\PaddleOCR-VL-1.6`.
Bucketed engine_config has no entry for resolved engine 'paddle_dynamic'; using an empty config for that engine.
Loading configuration file C:\Users\soura\.paddlex\official_models\PaddleOCR-VL-1.6\config.json
Loading weights file C:\Users\soura\.paddlex

PaddleOCR-VL 1.6 pipeline created


In [7]:
image_path = selected_images["scanning"]

start_time = time.perf_counter()

output = pipeline.predict(
    str(image_path)
)

elapsed = time.perf_counter() - start_time

print("Image:", image_path.name)
print("Inference time:", round(elapsed, 2), "seconds")
print("Output type:", type(output))

D:\mycode\GlyphForge\.venv\Lib\site-packages\paddle\tensor\creation.py:1152: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach(), rather than paddle.to_tensor(sourceTensor).
  return tensor(


Image: PPT_1001115_eng_page_003.png
Inference time: 259.37 seconds
Output type: <class 'list'>


In [8]:
print("Number of outputs:", len(output))

for index, item in enumerate(output):
    print(
        "Output",
        index,
        "| type:",
        type(item)
    )

Number of outputs: 1
Output 0 | type: <class 'paddlex.inference.pipelines.paddleocr_vl.result.PaddleOCRVLResult'>


In [9]:
first_output = output[0]

print(type(first_output))
print(first_output)

<class 'paddlex.inference.pipelines.paddleocr_vl.result.PaddleOCRVLResult'>
{'input_path': 'D:\\mycode\\GlyphForge\\data\\benchmark\\Real5-OmniDocBench-Scanning\\PPT_1001115_eng_page_003.png', 'page_index': None, 'page_count': None, 'width': 1677, 'height': 1186, 'doc_preprocessor_res': {'output_img': array([[[255, 255, 255],
        [255, 255, 255],
        [255, 255, 255],
        ...,
        [254, 254, 254],
        [253, 253, 253],
        [251, 251, 251]],

       [[255, 255, 255],
        [255, 255, 255],
        [255, 255, 255],
        ...,
        [254, 254, 254],
        [253, 253, 253],
        [251, 251, 251]],

       [[255, 255, 255],
        [255, 255, 255],
        [255, 255, 255],
        ...,
        [254, 254, 254],
        [253, 253, 253],
        [251, 251, 251]],

       ...,

       [[255, 255, 255],
        [255, 255, 255],
        [255, 255, 255],
        ...,
        [246, 246, 246],
        [234, 234, 234],
        [213, 213, 213]],

       [[255, 255, 255],

In [11]:
parsing_results = first_output["parsing_res_list"]

ocr_blocks = []

for block in parsing_results:
    ocr_blocks.append({
        "label": block.label,
        "content": block.content,
        "bbox": block.bbox
    })

ocr_blocks

[{'label': 'paragraph_title',
  'content': 'Who Am I?',
  'bbox': [608, 74, 1055, 166]},
 {'label': 'text',
  'content': '• Min-Te Sun (Peter) Sun',
  'bbox': [124, 251, 805, 329]},
 {'label': 'text',
  'content': '– An associate professor of Computer Science & Information Engineering, National Central University',
  'bbox': [195, 342, 1485, 503]},
 {'label': 'text',
  'content': '- Studied in US for a long time (from 1993 ~ 2002)',
  'bbox': [192, 491, 925, 638]},
 {'label': 'text',
  'content': '– Worked as a CS professor at Auburn University, Alabama between 2002 and 2008 (before coming back to Taiwan)',
  'bbox': [188, 642, 1525, 866]},
 {'label': 'text',
  'content': '— Have taught CS courses in English for more than 10 years',
  'bbox': [185, 861, 1434, 1004]}]

In [12]:
baseline_text = "\n".join(
    block["content"]
    for block in ocr_blocks
)

print(baseline_text)

Who Am I?
• Min-Te Sun (Peter) Sun
– An associate professor of Computer Science & Information Engineering, National Central University
- Studied in US for a long time (from 1993 ~ 2002)
– Worked as a CS professor at Auburn University, Alabama between 2002 and 2008 (before coming back to Taiwan)
— Have taught CS courses in English for more than 10 years


In [13]:
baseline_stats = {
    "category": "scanning",
    "image": image_path.name,
    "inference_time_seconds": elapsed,
    "block_count": len(ocr_blocks),
    "character_count": len(baseline_text),
    "word_count": len(baseline_text.split())
}

pd.DataFrame([baseline_stats])

,category,image,inference_time_seconds,block_count,character_count,word_count
0,scanning,PPT_1001115_eng_page_003.png,259.372122,6,354,64


In [14]:
baseline_output_path = (
    ocr_results_root
    / "baseline_scanning.json"
)

serializable_output = {
    "input_path": str(image_path),
    "inference_time_seconds": elapsed,
    "blocks": ocr_blocks,
    "text": baseline_text
}

with open(
    baseline_output_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        serializable_output,
        file,
        ensure_ascii=False,
        indent=2
    )

print(baseline_output_path)

D:\mycode\GlyphForge\results\ocr\baseline_scanning.json


In [15]:
def run_ocr(image_path):
    start_time = time.perf_counter()

    results = pipeline.predict(
        str(image_path)
    )

    elapsed = time.perf_counter() - start_time

    result = results[0]

    blocks = []

    for block in result["parsing_res_list"]:
        blocks.append({
            "label": block.label,
            "content": block.content,
            "bbox": block.bbox
        })

    text = "\n".join(
        block["content"]
        for block in blocks
    )

    return {
        "image": image_path.name,
        "inference_time_seconds": elapsed,
        "blocks": blocks,
        "text": text
    }

In [16]:
test_result = run_ocr(
    selected_images["scanning"]
)

print(
    "Image:",
    test_result["image"]
)

print(
    "Inference time:",
    round(
        test_result["inference_time_seconds"],
        2
    ),
    "seconds"
)

print(
    "Blocks:",
    len(test_result["blocks"])
)

print(
    "Characters:",
    len(test_result["text"])
)

print(
    "Words:",
    len(test_result["text"].split())
)

Image: PPT_1001115_eng_page_003.png
Inference time: 255.31 seconds
Blocks: 6
Characters: 354
Words: 64


In [17]:
baseline_results = []

for category, image_path in selected_images.items():
    print(
        "Running:",
        category,
        "|",
        image_path.name
    )

    result = run_ocr(image_path)

    baseline_results.append({
        "category": category,
        "image": result["image"],
        "inference_time_seconds": result[
            "inference_time_seconds"
        ],
        "block_count": len(result["blocks"]),
        "character_count": len(result["text"]),
        "word_count": len(
            result["text"].split()
        )
    })

    output_path = (
        ocr_results_root
        / f"baseline_{category}.json"
    )

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(
            {
                "category": category,
                "input_path": str(image_path),
                "inference_time_seconds": result[
                    "inference_time_seconds"
                ],
                "blocks": result["blocks"],
                "text": result["text"]
            },
            file,
            ensure_ascii=False,
            indent=2
        )

    print(
        "Finished:",
        round(
            result["inference_time_seconds"],
            2
        ),
        "seconds"
    )

Running: scanning | PPT_1001115_eng_page_003.png
Finished: 258.11 seconds
Running: warping | PPT_1001115_eng_page_003.png
Finished: 711.06 seconds
Running: screen_photography | PPT_1001115_eng_page_003.png
Finished: 623.81 seconds
Running: illumination | PPT_1001115_eng_page_003.png
Finished: 531.97 seconds
Running: skew | PPT_1001115_eng_page_003.png
Finished: 1659.72 seconds


In [18]:
baseline_results = pd.DataFrame(
    baseline_results
)

baseline_results

,category,image,inference_time_seconds,block_count,character_count,word_count
0,scanning,PPT_1001115_eng_page_003.png,258.106739,6,354,64
1,warping,PPT_1001115_eng_page_003.png,711.062288,6,354,64
2,screen_photography,PPT_1001115_eng_page_003.png,623.807097,6,354,64
3,illumination,PPT_1001115_eng_page_003.png,531.971367,6,354,64
4,skew,PPT_1001115_eng_page_003.png,1659.724175,6,352,63


In [19]:
baseline_benchmark_path = (
    ocr_results_root
    / "ocr_baseline.csv"
)

baseline_results.to_csv(
    baseline_benchmark_path,
    index=False
)

print(baseline_benchmark_path)

D:\mycode\GlyphForge\results\ocr\ocr_baseline.csv


In [20]:
ocr_compressed_root = ocr_results_root / "compressed"

ocr_generic_root = ocr_compressed_root / "jbig2_generic"
ocr_symbol_root = ocr_compressed_root / "jbig2_symbol"

ocr_generic_root.mkdir(
    parents=True,
    exist_ok=True
)

ocr_symbol_root.mkdir(
    parents=True,
    exist_ok=True
)

print(ocr_generic_root)
print(ocr_symbol_root)

D:\mycode\GlyphForge\results\ocr\compressed\jbig2_generic
D:\mycode\GlyphForge\results\ocr\compressed\jbig2_symbol


In [22]:
import cv2
import numpy as np

print("OpenCV:", cv2.__version__)

OpenCV: 4.10.0


In [23]:
for category in categories:
    decoded_path = (
        project_root
        / "results"
        / "images"
        / "compression"
        / "jbig2_decoded"
        / f"{category}.pbm"
    )

    output_path = (
        ocr_generic_root
        / f"{category}.png"
    )

    image = cv2.imread(
        str(decoded_path),
        cv2.IMREAD_GRAYSCALE
    )

    if image is None:
        raise RuntimeError(
            f"Could not read {decoded_path}"
        )

    success = cv2.imwrite(
        str(output_path),
        image
    )

    if not success:
        raise RuntimeError(
            f"Could not write {output_path}"
        )

    print(
        category,
        "|",
        image.shape,
        "|",
        round(
            output_path.stat().st_size / 1024,
            2
        ),
        "KB"
    )

scanning | (1186, 1677) | 43.5 KB
warping | (2160, 3840) | 143.24 KB
screen_photography | (2160, 3840) | 93.89 KB
illumination | (2160, 3840) | 43.22 KB
skew | (3072, 4096) | 322.68 KB


In [26]:
generic_verification = []

pbm_root = (
    project_root
    / "results"
    / "images"
    / "compression"
    / "pbm"
)

for category in categories:
    original_path = (
        pbm_root
        / f"{category}.pbm"
    )

    reconstructed_path = (
        ocr_generic_root
        / f"{category}.png"
    )

    original = cv2.imread(
        str(original_path),
        cv2.IMREAD_GRAYSCALE
    )

    reconstructed = cv2.imread(
        str(reconstructed_path),
        cv2.IMREAD_GRAYSCALE
    )

    if original is None:
        raise RuntimeError(
            f"Could not read original PBM: {original_path}"
        )

    if reconstructed is None:
        raise RuntimeError(
            f"Could not read reconstructed PNG: {reconstructed_path}"
        )

    if original.shape != reconstructed.shape:
        generic_verification.append({
            "category": category,
            "same_shape": False,
            "changed_pixels": None,
            "changed_pixel_percent": None,
            "exact_match": False
        })
        continue

    difference = cv2.compare(
        original,
        reconstructed,
        cv2.CMP_NE
    )

    changed_pixels = int(
        np.count_nonzero(difference)
    )

    generic_verification.append({
        "category": category,
        "same_shape": True,
        "changed_pixels": changed_pixels,
        "changed_pixel_percent": (
            changed_pixels / original.size * 100
        ),
        "exact_match": changed_pixels == 0
    })

generic_verification = pd.DataFrame(
    generic_verification
)

generic_verification

,category,same_shape,changed_pixels,changed_pixel_percent,exact_match
0,scanning,True,0,0.0,True
1,warping,True,0,0.0,True
2,screen_photography,True,0,0.0,True
3,illumination,True,0,0.0,True
4,skew,True,0,0.0,True


In [27]:
jbig2_symbol_root = (
    project_root
    / "results"
    / "images"
    / "compression"
    / "jbig2_symbol"
)

print(jbig2_symbol_root)

for category in categories:
    path = (
        jbig2_symbol_root
        / f"{category}.jb2"
    )

    print(
        category,
        "| exists:",
        path.exists(),
        "| size:",
        round(
            path.stat().st_size / 1024,
            2
        ) if path.exists() else None,
        "KB"
    )

D:\mycode\GlyphForge\results\images\compression\jbig2_symbol
scanning | exists: True | size: 5.89 KB
warping | exists: True | size: 33.58 KB
screen_photography | exists: True | size: 10.18 KB
illumination | exists: True | size: 6.96 KB
skew | exists: True | size: 139.67 KB


In [28]:
ocr_symbol_root = (
    ocr_compressed_root
    / "jbig2_symbol"
)

ocr_symbol_root.mkdir(
    parents=True,
    exist_ok=True
)

print(ocr_symbol_root)

D:\mycode\GlyphForge\results\ocr\compressed\jbig2_symbol


In [29]:
for category in categories:
    symbol_path = (
        jbig2_symbol_root
        / f"{category}.jb2"
    )

    if not symbol_path.exists():
        raise FileNotFoundError(
            symbol_path
        )

print("All Symbol JBIG2 files found")

All Symbol JBIG2 files found


In [31]:
import subprocess

print("subprocess ready")

subprocess ready


In [34]:
symbol_decoded_root = (
    project_root
    / "results"
    / "images"
    / "compression"
    / "jbig2_symbol_decoded"
)

symbol_decoded_root.mkdir(
    parents=True,
    exist_ok=True
)

for category in categories:
    symbol_path = (
        jbig2_symbol_root
        / f"{category}.jb2"
    )

    decoded_path = (
        symbol_decoded_root
        / f"{category}.pbm"
    )

    wsl_symbol_path = (
        "/mnt/d/mycode/GlyphForge/"
        "results/images/compression/"
        "jbig2_symbol/"
        f"{category}.jb2"
    )

    wsl_decoded_path = (
        "/mnt/d/mycode/GlyphForge/"
        "results/images/compression/"
        "jbig2_symbol_decoded/"
        f"{category}.pbm"
    )

    command = [
        "wsl",
        "-d",
        "Ubuntu",
        "/usr/bin/jbig2dec",
        "-o",
        wsl_decoded_path,
        "-t",
        "pbm",
        wsl_symbol_path
    ]

    result = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )

    if result.returncode != 0:
        raise RuntimeError(
            result.stderr.decode(
                "utf-8",
                errors="replace"
            )
        )

    print(
        category,
        "| decoded:",
        decoded_path.exists()
    )

scanning | decoded: True
warping | decoded: True
screen_photography | decoded: True
illumination | decoded: True
skew | decoded: True


In [35]:
for category in categories:
    decoded_path = (
        symbol_decoded_root
        / f"{category}.pbm"
    )

    print(
        category,
        "| exists:",
        decoded_path.exists(),
        "| size:",
        round(
            decoded_path.stat().st_size / 1024,
            2
        ) if decoded_path.exists() else None,
        "KB"
    )

scanning | exists: True | size: 243.24 KB
warping | exists: True | size: 1012.51 KB
screen_photography | exists: True | size: 1012.51 KB
illumination | exists: True | size: 1012.51 KB
skew | exists: True | size: 1536.01 KB


In [36]:
symbol_verification = []

for category in categories:
    original_path = (
        pbm_root
        / f"{category}.pbm"
    )

    reconstructed_path = (
        symbol_decoded_root
        / f"{category}.pbm"
    )

    original = cv2.imread(
        str(original_path),
        cv2.IMREAD_GRAYSCALE
    )

    reconstructed = cv2.imread(
        str(reconstructed_path),
        cv2.IMREAD_GRAYSCALE
    )

    if original is None:
        raise RuntimeError(
            f"Could not read original PBM: {original_path}"
        )

    if reconstructed is None:
        raise RuntimeError(
            f"Could not read Symbol decoded PBM: {reconstructed_path}"
        )

    if original.shape != reconstructed.shape:
        symbol_verification.append({
            "category": category,
            "same_shape": False,
            "changed_pixels": None,
            "changed_pixel_percent": None,
            "exact_match": False
        })
        continue

    difference = cv2.compare(
        original,
        reconstructed,
        cv2.CMP_NE
    )

    changed_pixels = int(
        np.count_nonzero(difference)
    )

    symbol_verification.append({
        "category": category,
        "same_shape": True,
        "changed_pixels": changed_pixels,
        "changed_pixel_percent": (
            changed_pixels / original.size * 100
        ),
        "exact_match": changed_pixels == 0
    })

symbol_verification = pd.DataFrame(
    symbol_verification
)

symbol_verification

,category,same_shape,changed_pixels,changed_pixel_percent,exact_match
0,scanning,True,682,0.034290,False
1,warping,True,461,0.005558,False
2,screen_photography,True,2082,0.025101,False
3,illumination,True,0,0.000000,True
4,skew,True,592,0.004705,False


In [37]:
for category in categories:
    decoded_path = (
        symbol_decoded_root
        / f"{category}.pbm"
    )

    output_path = (
        ocr_symbol_root
        / f"{category}.png"
    )

    image = cv2.imread(
        str(decoded_path),
        cv2.IMREAD_GRAYSCALE
    )

    if image is None:
        raise RuntimeError(
            f"Could not read {decoded_path}"
        )

    success = cv2.imwrite(
        str(output_path),
        image
    )

    if not success:
        raise RuntimeError(
            f"Could not write {output_path}"
        )

    print(
        category,
        "|",
        image.shape,
        "|",
        round(
            output_path.stat().st_size / 1024,
            2
        ),
        "KB"
    )

scanning | (1186, 1677) | 43.49 KB
warping | (2160, 3840) | 143.25 KB
screen_photography | (2160, 3840) | 93.96 KB
illumination | (2160, 3840) | 43.22 KB
skew | (3072, 4096) | 322.67 KB


In [38]:
for category in categories:
    path = (
        ocr_symbol_root
        / f"{category}.png"
    )

    print(
        category,
        "| exists:",
        path.exists(),
        "| size:",
        round(
            path.stat().st_size / 1024,
            2
        ) if path.exists() else None,
        "KB"
    )

scanning | exists: True | size: 43.49 KB
warping | exists: True | size: 143.25 KB
screen_photography | exists: True | size: 93.96 KB
illumination | exists: True | size: 43.22 KB
skew | exists: True | size: 322.67 KB


In [40]:
generic_ocr_results = []

for category in categories:
    image_path = (
        ocr_generic_root
        / f"{category}.png"
    )

    print(
        "Running Generic JBIG2 OCR:",
        category
    )

    result = run_ocr(image_path)

    generic_ocr_results.append({
        "category": category,
        "image": result["image"],
        "inference_time_seconds": result[
            "inference_time_seconds"
        ],
        "block_count": len(result["blocks"]),
        "character_count": len(result["text"]),
        "word_count": len(
            result["text"].split()
        )
    })

    output_path = (
        ocr_results_root
        / f"generic_jbig2_{category}.json"
    )

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(
            {
                "category": category,
                "input_path": str(image_path),
                "inference_time_seconds": result[
                    "inference_time_seconds"
                ],
                "blocks": result["blocks"],
                "text": result["text"]
            },
            file,
            ensure_ascii=False,
            indent=2
        )

    print(
        "Finished:",
        round(
            result["inference_time_seconds"],
            2
        ),
        "seconds"
    )

Running Generic JBIG2 OCR: scanning
Finished: 360.2 seconds
Running Generic JBIG2 OCR: warping
Finished: 953.89 seconds
Running Generic JBIG2 OCR: screen_photography


VLM worker did not terminate in time


KeyboardInterrupt: 

In [ ]:
generic_ocr_results = pd.DataFrame(
    generic_ocr_results
)

generic_ocr_results